# Kelly Criterion Deep Dive

This notebook explores Kelly criterion variants beyond the binary formula: multi-outcome Kelly,
fractional Kelly, Kelly under edge uncertainty, and correlated-bet Kelly. Each variant is
demonstrated with concrete parameter sweeps and visualizations of the tradeoff between growth
rate and ruin probability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from speculator_ev_engine.core.kelly import (
    binary_kelly, fractional_kelly, multi_outcome_kelly,
    correlated_kelly, uncertain_edge_kelly,
)

## 1. Binary Kelly: fraction vs edge

In [ ]:
edges = np.linspace(-0.05, 0.15, 50)
fractions = []
log_growths = []
ruin_probs = []

for edge in edges:
    # edge = p*b - q, solve for p given b=1 (even money)
    # p*(1+b) - 1 = edge => p = (1 + edge) / 2
    p = (1.0 + edge) / 2.0
    p = np.clip(p, 0.01, 0.99)
    result = binary_kelly(p, b=1.0)
    fractions.append(result.fraction)
    log_growths.append(result.expected_log_growth)
    ruin_probs.append(result.ruin_probability)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(edges, fractions)
axes[0].set_title('Kelly Fraction vs Edge')
axes[0].set_xlabel('Edge')
axes[0].axhline(0, color='gray', linestyle='--')

axes[1].plot(edges, log_growths)
axes[1].set_title('Expected Log Growth vs Edge')
axes[1].set_xlabel('Edge')
axes[1].axhline(0, color='gray', linestyle='--')

axes[2].plot(edges, ruin_probs)
axes[2].set_title('Ruin Probability vs Edge')
axes[2].set_xlabel('Edge')

plt.tight_layout()
plt.show()

## 2. Fractional Kelly: growth-variance tradeoff

In [ ]:
# p=0.55, b=1.0 (5% edge on even money)
p = 0.55
b = 1.0
fracs = np.linspace(0.05, 1.0, 20)
results = [fractional_kelly(p, b, fraction=f) for f in fracs]

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(fracs, [r.expected_log_growth for r in results], 'b-o', label='Log Growth')
ax1.set_xlabel('Fraction of Full Kelly')
ax1.set_ylabel('Expected Log Growth', color='b')
ax2 = ax1.twinx()
ax2.plot(fracs, [r.ruin_probability for r in results], 'r-s', label='Ruin Prob')
ax2.set_ylabel('Ruin Probability', color='r')
plt.title('Fractional Kelly: Growth vs Ruin Tradeoff (p=0.55, b=1.0)')
plt.show()

## 3. Multi-Outcome Kelly: 3-way market example

In [ ]:
# Soccer 1X2 market with positive EV on home win
probs = np.array([0.50, 0.30, 0.20])  # model estimates
payouts = np.array([1.2, -1.0, -1.0])  # home at +120, lose 1u otherwise
result = multi_outcome_kelly(probs, payouts)
print(f'Optimal fraction: {result.fraction:.4f}')
print(f'Expected log growth: {result.expected_log_growth:.6f}')
print(f'Ruin probability: {result.ruin_probability:.6f}')

## 4. Kelly Under Edge Uncertainty

In [ ]:
# How does Kelly fraction shrink as edge uncertainty grows?
edge_mean = 0.05
odds = 1.0
stds = np.linspace(0.0, 0.10, 20)
fractions = []
for std in stds:
    result = uncertain_edge_kelly(edge_mean, std, odds, n_samples=5000, seed=42)
    fractions.append(result.fraction)

plt.figure(figsize=(10, 6))
plt.plot(stds, fractions, 'k-o')
plt.xlabel('Edge Standard Deviation')
plt.ylabel('Median Kelly Fraction')
plt.title('Kelly Shrinks as Edge Uncertainty Grows')
plt.axhline(binary_kelly(0.525, 1.0).fraction, color='gray', linestyle='--', label='No uncertainty')
plt.legend()
plt.show()

## 5. Correlated Kelly: two simultaneous bets

In [ ]:
# Two NFL spreads with moderate correlation
edges = np.array([0.03, 0.04])
odds = np.array([1.0, 1.0])
corr_values = [0.0, 0.3, 0.6, 0.9]

for corr in corr_values:
    cov = np.array([[1.0, corr], [corr, 1.0]])
    result = correlated_kelly(edges, odds, cov)
    print(f'Correlation={corr:.1f}: total fraction={result.fraction:.4f}, '
          f'log growth={result.expected_log_growth:.6f}, ruin={result.ruin_probability:.4f}')